# Frameworks 00 - LangGraph

Objetivo: ejecutar el adaptador LangGraph real y un StateGraph condicional,
manteniendo Provider, Framework, estado, resultado nativo y lineage observables.


**Lugar en el modelo:** LangGraph es el Framework que controla el grafo/loop; el Provider sigue determinando dónde ocurre la inferencia.

**Evidencia exigida:** el SDK real debe compilar y ejecutar rutas sync/async y un grafo condicional, normalizadas a `RunResult`/estado observable.

**Límite de la evidencia:** `python-runtime` aporta un modelo determinista al adaptador; no significa que LangGraph ejecute por sí solo un pipeline arbitrario de Agentic Systems.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| Provider offline | python-runtime | Inferencia determinista sin red. |
| Provider live | auto | Ruta opcional con RUN_LANGGRAPH_LIVE=1. |
| Framework | langgraph | Compila y ejecuta el SDK real. |


## 1) Provider y Framework independientes


In [ ]:
import os

import agentic_systems as toolkit

RUN_LIVE = os.getenv("RUN_LANGGRAPH_LIVE", "0").strip().lower() in {"1", "true", "yes"}
runtime = (
    toolkit.runtime(
        provider="auto",
        provider_priority=["openai-runtime", "vllm-runtime", "bedrock-runtime"],
    )
    if RUN_LIVE
    else toolkit.runtime(provider="python-runtime")
)
framework = toolkit.framework("langgraph")
profile = toolkit.integrations.framework_profile("langgraph")
runtime_description = runtime.describe()
toolkit.show_json(
    {"runtime": runtime_description, "framework": framework.inspect(), "profile": profile.to_dict()},
    title="Provider x LangGraph",
)


## 2) Agent ejecutado dentro de un grafo real de un nodo


In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

agent = toolkit.agent(
    name="langgraph_inspector",
    instructions="Ejecuta inspect_public_api y conserva la evidencia.",
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
agent.prepare()
sync_result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "graph"}},
    mode="eval",
)
async_result = await agent.arun(
    {"tool": "inspect_public_api", "input": {"symbol": "framework"}},
    mode="eval",
)
assert sync_result.ok and async_result.ok
assert sync_result.engine == async_result.engine == runtime_description["selected_provider"]
assert sync_result.meta["framework_adapter"] == "langgraph"
assert async_result.meta["framework_adapter"] == "langgraph"
toolkit.human_result(sync_result, title="LangGraph Agent RunResult", show_lineage=True)
toolkit.show_json(
    {
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(sync_result.native_result).__name__,
        "sync_framework": sync_result.meta["framework_adapter"],
        "async_framework": async_result.meta["framework_adapter"],
    },
    title="Real one-node StateGraph",
)


## 3) Estado, routing y ramas con toolkit.graph


In [ ]:
def classify(state: dict) -> dict:
    return {**state, "route": "accepted" if state["score"] >= 0.5 else "rejected"}

def accept(state: dict) -> dict:
    return {**state, "decision": "accepted"}

def reject(state: dict) -> dict:
    return {**state, "decision": "rejected"}

def route(state: dict) -> str:
    return state["route"]

app = toolkit.graph(
    name="langgraph_routing",
    engine="langgraph",
    state=dict,
    nodes={"classify": classify, "accept": accept, "reject": reject},
    edges=[("START", "classify"), ("accept", "END"), ("reject", "END")],
    conditional_edges=[
        ("classify", route, {"accepted": "accept", "rejected": "reject"}),
    ],
)
accepted_state = app.run({"score": 0.9})
rejected_state = await app.arun({"score": 0.1})
assert accepted_state["decision"] == "accepted"
assert rejected_state["decision"] == "rejected"
toolkit.show_json(
    {
        "engine": app.engine,
        "framework": app.framework,
        "native_graph": type(app.native).__name__,
        "accepted": accepted_state,
        "rejected": rejected_state,
    },
    title="Conditional StateGraph",
)


## 4) Lineage desde el estado ejecutado


In [ ]:
lineage = app.lineage(
    accepted_state,
    question="Que rama eligio LangGraph?",
    goal="Conservar estado, routing y decision.",
    answer_keys=("decision", "route"),
)
toolkit.show(lineage, title="LangGraph lineage")

api_coverage = [
    "toolkit.runtime", "toolkit.framework", "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.AgentContract",
    "toolkit.graph", "GraphApp.run", "GraphApp.arun", "GraphApp.lineage",
    "toolkit.human_result", "toolkit.show", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="LangGraph API coverage")


## Resultado e interpretacion

El SDK compila un grafo de agente y un grafo condicional. Las rutas sync/async,
el estado nativo y lineage permanecen observables sin mezclar LangGraph con core.
